<a href="https://colab.research.google.com/github/WhoisMonesh/Torrent2Gdrive/blob/main/Torrent_To_Google_Drive_Downloader_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Torrent To Google Drive Downloader v3

### Features
- Download torrents/magnets directly to Google Drive
- File selection: pick which files to download
- Sequential download (stream while downloading)
- Progress bar with ETA
- Resume support
- Speed limits
- Supports both magnet links and .torrent file uploads

### 1. Mount Google Drive

In [0]:
from google.colab import drive
drive.mount('/content/drive')

### 2. Install Dependencies
[libtorrent](https://www.libtorrent.org/) is the BitTorrent library used.

In [0]:
!pip install libtorrent tqdm -q
print('Dependencies installed.')

### 3. Configuration
Edit the variables below before running the download cell.

In [0]:
# === CONFIGURATION ===
SAVE_PATH = '/content/downloads/Torrent/'  # local temp dir (avoids Drive sync conflicts)
DRIVE_PATH = '/content/drive/My Drive/Torrent/'  # final destination on Drive

# Paste your magnet link here, or leave empty to upload a .torrent file
MAGNET_LINK = ""  # e.g., "magnet:?xt=urn:btih:..."

# File selection: set to True to pick specific files from the torrent
SELECT_FILES = False

# Sequential download: download in order (useful for streaming video)
SEQUENTIAL_DOWNLOAD = False

# Speed limits (0 = unlimited)
DOWNLOAD_SPEED_LIMIT = 0       # bytes per second
UPLOAD_SPEED_LIMIT = 0         # bytes per second

# Max connections
MAX_CONNECTIONS = 200
ZIP_PATH = ''  # set to create a zip archive (auto-zips when multiple files)

# Prevent Colab timeout: clicks reconnect button every 2 min
KEEP_ALIVE = True  # set to False to disable

# Extra trackers to boost peer discovery
EXTRA_TRACKERS = [
        "udp://tracker.opentrackr.org:1337/announce",
        "udp://tracker.torrent.eu.org:451/announce",
        "udp://open.stealth.si:80/announce",
        "https://tracker.tamersunion.org:443/announce",
        "udp://exodus.desync.com:6969/announce",
        "udp://tracker.moeking.me:6969/announce",
        "udp://tracker.dler.org:6969/announce",
        "udp://tracker.zer0day.to:1337/announce",
        "udp://coppersurfer.tk:6969/announce",
        "https://tracker.nanoha.org:443/announce",
    ]
# === END CONFIGURATION ===

### 4. Download Torrent
Run this cell to start the download.

In [0]:
!pip install libtorrent tqdm -q 2>/dev/null

import libtorrent as lt
import time
import datetime
import os
import shutil
from google.colab import files

lt_version = lt.__version__ if hasattr(lt, '__version__') else '1.x'
print(f'libtorrent version: {lt_version}')

SEEDING = lt.torrent_status.states.seeding if hasattr(lt.torrent_status, 'states') else lt.torrent_status.seeding
DOWNLOADING = lt.torrent_status.states.downloading if hasattr(lt.torrent_status, 'states') else lt.torrent_status.downloading

def setup_session():
    settings = {
        'listen_interfaces': '0.0.0.0:6881',
        'dht_bootstrap_nodes': 'dht.libtorrent.org:25401,router.bittorrent.com:6881,dht.transmissionbt.com:6881,router.utorrent.com:6881',
        'user_agent': 'Torrent2GDrive/v3',
        'connections_limit': MAX_CONNECTIONS,
    }
    if hasattr(lt.alert.category_t, 'error_notification'):
        settings['alert_mask'] = lt.alert.category_t.error_notification
    ses = lt.session(settings)
    return ses

def add_torrent(ses, link, save_path):
    if link.startswith('magnet:'):
        print('Adding magnet link...')
        params = lt.parse_magnet_uri(link)
    else:
        print('Adding torrent file...')
        params = lt.add_torrent_params()
        params.ti = lt.torrent_info(link)
    params.save_path = save_path
    params.storage_mode = lt.storage_mode_t.storage_mode_sparse
    handle = ses.add_torrent(params)
    return handle

def wait_for_metadata(handle, timeout=120):
    print('Downloading metadata...')
    start = time.time()
    while not handle.status().has_metadata:
        if time.time() - start > timeout:
            print('Metadata taking longer than expected, still waiting...')
            timeout += 120
        time.sleep(1)
    print('Got metadata!')

def select_files(handle):
    if hasattr(handle, 'torrent_info'):
        tinfo = handle.torrent_info()
    else:
        tinfo = handle.torrent_file()
    files = tinfo.files()
    print(f'\nTorrent contains {files.num_files()} files:')
    for i in range(files.num_files()):
        size_mb = files.file_size(i) / (1024 * 1024)
        print(f'  [{i}] {files.file_name(i)} ({size_mb:.1f} MB)')
    indices = input('\nEnter file indices to download (comma-separated, e.g. 0,2,5), or press Enter for all: ')
    if indices.strip():
        selected = [int(x.strip()) for x in indices.split(',')]
        for i in range(files.num_files()):
            if i not in selected:
                handle.file_priority(i, 0)
        print(f'Downloading files: {selected}')

def format_bytes(n):
    for unit in ['B', 'KB', 'MB', 'GB', 'TB']:
        if n < 1024:
            return f'{n:.1f} {unit}'
        n /= 1024
    return f'{n:.1f} PB'


def add_extra_trackers(handle, trackers):
    for i, url in enumerate(trackers):
        handle.add_tracker({'url': url, 'tier': i // 2})
        print(f'  + {url}')
    handle.force_reannounce()
    print('Tracker announce complete.')
def main():
    os.makedirs(SAVE_PATH, exist_ok=True)

    link = MAGNET_LINK.strip()
    if not link:
        print('No magnet link provided. Please upload a .torrent file.')
        uploaded = files.upload()
        if not uploaded:
            print('No file uploaded. Exiting.')
            return
        link = list(uploaded.keys())[0]
        print(f'Using torrent file: {link}')

    print(f'Save path: {SAVE_PATH} (local, avoids Drive sync conflicts)')
    print('Setting up session...')
    ses = setup_session()

    print('Adding torrent...')
    handle = add_torrent(ses, link, SAVE_PATH)

    if link.startswith('magnet:'):
        try:
            wait_for_metadata(handle)
        except TimeoutError as e:
            print(f'Error: {e}')
            return

    if SELECT_FILES:
        select_files(handle)

    if DOWNLOAD_SPEED_LIMIT > 0:
        ses.download_rate_limit(DOWNLOAD_SPEED_LIMIT)
    if UPLOAD_SPEED_LIMIT > 0:
        ses.upload_rate_limit(UPLOAD_SPEED_LIMIT)


    if EXTRA_TRACKERS:
        print(f'\nAdding {len(EXTRA_TRACKERS)} extra trackers...')
        add_extra_trackers(handle, EXTRA_TRACKERS)
        print()
    if SEQUENTIAL_DOWNLOAD:
        handle.set_sequential_download(True)
        print('Sequential download enabled')

    print(f'\nStarting: {handle.status().name}')
    print(f'Total size: {format_bytes(handle.status().total_wanted)}')
    print(f'Started at: {datetime.datetime.now()}')

    if KEEP_ALIVE:
        from IPython.display import display, Javascript
        display(Javascript("""
          function keepAlive() {
            var btn = document.querySelector('colab-connect-button');
            if (btn) btn.click();
          }
          setInterval(keepAlive, 120000);
          console.log('Keep-alive active');
        print('Keep-alive active - session will stay awake')
        """))

    begin = time.time()
    state_str = ['queued', 'checking', 'downloading metadata',
                 'downloading', 'finished', 'seeding', 'allocating']
    last_print = 0
    from IPython.display import display, HTML
    progress_display = display(HTML('<pre>Waiting for progress...</pre>'), display_id='dl-progress')

    try:
        while handle.status().state != SEEDING:
            s = handle.status()
            progress = int(s.progress * 100)
            if progress != last_print:
                last_print = progress
                status = state_str[s.state]
                down = s.download_rate / 1000
                up = s.upload_rate / 1000
                total = format_bytes(s.total_wanted)
                downloaded = format_bytes(s.total_download)
                bar = '#' * (progress // 2) + '-' * (50 - progress // 2)
                progress_display.update(HTML(f'<pre>{status}: |{bar}| {progress}% | D:{down:.1f} U:{up:.1f} kB/s | {downloaded}/{total} | peers:{s.num_peers}</pre>'))
            time.sleep(2)
    except KeyboardInterrupt:
        print('\n\nDownload paused.')
    finally:
        pass
    end = time.time()
    elapsed = end - begin
    mins = int(elapsed // 60)
    secs = int(elapsed % 60)

    torrent_name = handle.status().name
    print(f'\n{"="*50}')
    print(f'{torrent_name} COMPLETE')
    print(f'Elapsed Time: {mins} min : {secs} sec')
    print(f'Average Speed: {handle.status().total_download / (elapsed or 1) / 1024:.1f} kB/s')
    print(f'Finished at: {datetime.datetime.now()}')
    print(f'Saved locally: {SAVE_PATH}')

    # Count total files (recursive) to decide if zip is needed
    all_items = []
    for root_dir, _, filenames in os.walk(SAVE_PATH):
        for f in filenames:
            all_items.append(os.path.join(root_dir, f))
    file_count = len(all_items)

    zip_target = ''
    if ZIP_PATH:
        zip_target = ZIP_PATH
    elif file_count > 1:
        zip_target = os.path.join(SAVE_PATH, torrent_name.replace(' ', '_').replace(':', '-')) + '.zip'
        print(f'\nAuto-zip: {file_count} files detected')

    if zip_target:
        import zipfile
        total_size = sum(os.path.getsize(fp) for fp in all_items)
        processed = 0
        os.makedirs(os.path.dirname(zip_target), exist_ok=True)
        print('Creating zip archive...')
        with zipfile.ZipFile(zip_target, 'w', zipfile.ZIP_DEFLATED) as zf:
            for fp in all_items:
                arcname = os.path.relpath(fp, SAVE_PATH)
                zf.write(fp, arcname)
                processed += os.path.getsize(fp)
                pct = int(processed * 100 / total_size) if total_size > 0 else 0
                bar = '#' * (pct // 2) + '-' * (50 - pct // 2)
                progress_display.update(HTML(f'<pre>Zipping: |{bar}| {pct}% | {arcname}</pre>'))
        print(f'Zip: {os.path.basename(zip_target)} ({format_bytes(os.path.getsize(zip_target))})')
        from IPython.display import display as ipd_display
        ipd_display(HTML(f'<a href="{zip_target}" download>Click to download ZIP</a>'))
        files.download(zip_target)

    print(f'\nMoving completed files to Drive (avoids sync conflicts)...')
    os.makedirs(DRIVE_PATH, exist_ok=True)
    move_count = 0
    for f in os.listdir(SAVE_PATH):
        shutil.move(os.path.join(SAVE_PATH, f), os.path.join(DRIVE_PATH, f))
        move_count += 1
    print(f'Moved {move_count} items')
    print(f'Final location: {DRIVE_PATH}')
    print(f'{"="*50}')
if __name__ == '__main__':
    main()


### 5. (Optional) Resume Incomplete Download
If the download was interrupted, run this cell alone to resume it (requires the same magnet/torrent).

In [0]:
# Resumes fast-resume data for any incomplete torrents in the save path
# Run the download cell above again after this
print('Incomplete torrents will auto-resume when re-added to the session.')
print('Just run the download cell again with the same magnet/torrent.')